#Import

In [ ]:
from __future__ import annotations
from typing import Optional, Dict, Callable
from pathlib import Path
from copy import deepcopy
from collections import Counter

import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from scipy.stats import chi2_contingency
from graphviz import Digraph, Source
from IPython.display import display as idisplay
from matplotlib import pyplot as plt
import seaborn as sns
from tqdm import tqdm

In [ ]:
education_train_set = pd.read_csv('/content/Data/CC_education_economy_train.csv')
education_test_set = pd.read_csv('/content/Data/CC_education_economy_test.csv')

numerical_continuous_attributes = ['aggregated_score', 'salary', 'experience_years', 'skills_count', 'certifications', 'total_days_worked']
categoric_attributes = ['job_title', 'industry', 'remote_work', 'location', 'vacation']
ordinal_attributes = ['education_level', 'company_size', 'skill_bracket']
categoric_ordinal_attributes = ['job_title', 'industry', 'remote_work', 'location', 'vacation','education_level', 'company_size', 'skill_bracket']

education_train_set[numerical_continuous_attributes].describe()

FileNotFoundError: [Errno 2] No such file or directory: '/content/Data/CC_education_economy_train.csv'

In [ ]:
for attribute in numerical_continuous_attributes:
    education_train_set.boxplot(column=attribute)
    plt.title(f'Box Plot of {attribute}')
    plt.ylabel(attribute)
    plt.show()

In [ ]:
def get_discrete_ordinal_attributes(dataset, attributes):
    tables = []

    for attribute in attributes:
        attribute_characteristics = (dataset[attribute].count(), dataset[attribute].dropna().unique().shape[0])

        tables.append(pd.DataFrame({'count': attribute_characteristics[0], 'unique': attribute_characteristics[1]}, index = [attribute]))

    return pd.concat(tables)


In [ ]:
discrete_ordinal_attributes = get_discrete_ordinal_attributes(education_train_set, categoric_ordinal_attributes)
discrete_ordinal_attributes

In [ ]:
def plot_discrete_ordinal_attributes(dataset, attributes, widths):
    for (attribute, width) in zip(attributes, widths):
        plt.figure(figsize=(10, 6)) # Set a larger figure size
        dataset[attribute].hist(bins=dataset[attribute].nunique(), width=width)
        plt.xlabel(attribute)
        plt.ylabel('Frequency')
        plt.title(f'Distribution of {attribute}') # More descriptive title
        plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for better readability
        plt.tight_layout() # Adjust layout to prevent labels from being cut off
        plt.show()

In [ ]:
plot_discrete_ordinal_attributes(education_train_set,
                                categoric_ordinal_attributes,
                                [
                                    0.2, 0.2, 0.5, 0.2, 0.2,
                                    0.2, 0.2, 0.2
                                ])

In [ ]:
correlations = education_train_set[numerical_continuous_attributes].corr(method='pearson')
fig = plt.figure(figsize=(10,10))

# 111: 1x1 grid, first subplot
ax = fig.add_subplot(111)

# normalize data using vmin, vmax
cax = ax.matshow(correlations, vmin=-1, vmax=1)

# add a colorbar to a plot.
fig.colorbar(cax)

# define ticks based on the number of numerical_continuous_attributes
ticks = np.arange(len(numerical_continuous_attributes))

# set x and y tick marks
ax.set_xticks(ticks)
ax.set_yticks(ticks)

# set x and y tick labels
ax.set_xticklabels(numerical_continuous_attributes, rotation=90)
ax.set_yticklabels(numerical_continuous_attributes)

# draw a matrix using the correlations data
plt.show()

In [ ]:
def get_corr_matrix_discrete_ordinal_attributes(dataset, attributes):
    matrix = pd.DataFrame(index=attributes, columns=attributes)

    for i in range(len(attributes)):
        for j in range(i, len(attributes)):
            if i == j:
                matrix.at[attributes[i], attributes[j]] = 1
                continue

            attribute1 = attributes[i]
            attribute2 = attributes[j]

            table = pd.crosstab(dataset[attribute1], dataset[attribute2], margins=False)
            _, p, _, _ = chi2_contingency(table)
            matrix.at[attribute1, attribute2] = 1 - p
            matrix.at[attribute2, attribute1] = 1 - p

    return matrix.astype(float)

In [ ]:
categoric_attributes_for_corr_matrix = categoric_ordinal_attributes
corr_matrix_discrete_ordinal_attributes = get_corr_matrix_discrete_ordinal_attributes(education_train_set, categoric_attributes_for_corr_matrix)

corr_matrix_discrete_ordinal_attributes

In [ ]:
fig = plt.figure(figsize=(12,10))

# 111: 1x1 grid, first subplot
ax = fig.add_subplot(111)

# normalize data using vmin, vmax
cax = ax.matshow(corr_matrix_discrete_ordinal_attributes, vmin=-1, vmax=1)

# add a colorbar to a plot.
fig.colorbar(cax)

# define ticks for all columns using the correlation matrix's index/columns
attribute_labels = corr_matrix_discrete_ordinal_attributes.columns.tolist()
ticks = np.arange(len(attribute_labels))

# set x and y tick marks
ax.set_xticks(ticks)
ax.set_yticks(ticks)

# set x and y tick labels
ax.set_xticklabels(attribute_labels, rotation=90)
ax.set_yticklabels(attribute_labels)

# Add a title to the plot
plt.title('Correlation Matrix of Discrete Ordinal Attributes (Chi-squared based, including vacation)')

# draw a matrix using the correlations data
plt.show()

Preprocesarea Datelor

Observ ca doar remote_work are date lipsa.

In [ ]:
education_train_processed_categorical_imputed = education_train_set.copy()
education_test_processed_categorical_imputed = education_test_set.copy()

categorical_features_to_impute = [col for col in categoric_attributes + ordinal_attributes if col != 'vacation']
categorical_imputer = SimpleImputer(strategy='most_frequent')

education_train_processed_categorical_imputed[categorical_features_to_impute] = categorical_imputer.fit_transform(education_train_processed_categorical_imputed[categorical_features_to_impute])
education_test_processed_categorical_imputed[categorical_features_to_impute] = categorical_imputer.transform(education_test_processed_categorical_imputed[categorical_features_to_impute])

In [ ]:
education_train_processed_outliers_handled = education_train_processed_categorical_imputed.copy()
education_test_processed_outliers_handled = education_test_processed_categorical_imputed.copy()

for attribute in numerical_continuous_attributes:
    Q1 = education_train_processed_outliers_handled[attribute].quantile(0.25)
    Q3 = education_train_processed_outliers_handled[attribute].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    education_train_processed_outliers_handled[attribute] = education_train_processed_outliers_handled[attribute].apply(
        lambda x: np.nan if x < lower_bound or x > upper_bound else x
    )
    education_test_processed_outliers_handled[attribute] = education_test_processed_outliers_handled[attribute].apply(
        lambda x: np.nan if x < lower_bound or x > upper_bound else x
    )

In [ ]:
for attribute in numerical_continuous_attributes:
    imputer = SimpleImputer(strategy='mean')
    education_train_processed_outliers_handled[[attribute]] = imputer.fit_transform(education_train_processed_outliers_handled[[attribute]])
    education_test_processed_outliers_handled[[attribute]] = imputer.transform(education_test_processed_outliers_handled[[attribute]])

In [ ]:
education_train_processed_final = education_train_processed_outliers_handled.copy()
education_test_processed_final = education_test_processed_outliers_handled.copy()

scaler = StandardScaler()
education_train_processed_final[numerical_continuous_attributes] = scaler.fit_transform(education_train_processed_final[numerical_continuous_attributes])
education_test_processed_final[numerical_continuous_attributes] = scaler.transform(education_test_processed_final[numerical_continuous_attributes])

Utilizarea algoritmilor de invatare automata

### Clasificare

### Model Evaluation on Dataset with Categorical Imputation Only

In [ ]:
# Prepare data for the 'Categorical Imputation Only' stage
X_train_stage1_raw = education_train_processed_categorical_imputed.drop('vacation', axis=1)
y_train_stage1 = education_train_processed_categorical_imputed['vacation']
X_test_stage1_raw = education_test_processed_categorical_imputed.drop('vacation', axis=1)
y_test_stage1 = education_test_processed_categorical_imputed['vacation']

# Define categorical features for One-Hot Encoding
categorical_features_for_ohe = [col for col in categoric_ordinal_attributes if col != 'vacation']

# Apply One-Hot Encoding
ohe_stage1 = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_stage1_encoded = ohe_stage1.fit_transform(X_train_stage1_raw[categorical_features_for_ohe])
X_train_stage1_encoded_df = pd.DataFrame(X_train_stage1_encoded, columns=ohe_stage1.get_feature_names_out(categorical_features_for_ohe), index=X_train_stage1_raw.index)
X_train_stage1_processed = X_train_stage1_raw.drop(columns=categorical_features_for_ohe)
X_train_stage1_processed = pd.concat([X_train_stage1_processed.reset_index(drop=True), X_train_stage1_encoded_df.reset_index(drop=True)], axis=1)

X_test_stage1_encoded = ohe_stage1.transform(X_test_stage1_raw[categorical_features_for_ohe])
X_test_stage1_encoded_df = pd.DataFrame(X_test_stage1_encoded, columns=ohe_stage1.get_feature_names_out(categorical_features_for_ohe), index=X_test_stage1_raw.index)
X_test_stage1_processed = X_test_stage1_raw.drop(columns=categorical_features_for_ohe)
X_test_stage1_processed = pd.concat([X_test_stage1_processed.reset_index(drop=True), X_test_stage1_encoded_df.reset_index(drop=True)], axis=1)

# Train and evaluate Decision Tree Classifier
dt_classifier_stage1 = DecisionTreeClassifier()
dt_classifier_stage1.fit(X_train_stage1_processed, y_train_stage1)
y_pred_stage1 = dt_classifier_stage1.predict(X_test_stage1_processed)

print("--- Results for Categorical Imputation Only ---")
print(f"Accuracy: {accuracy_score(y_test_stage1, y_pred_stage1):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_stage1, y_pred_stage1))

### Model Evaluation on Dataset with Outlier Handling and Numerical Imputation

In [ ]:
# Prepare data for the 'Outlier Handling and Numerical Imputation' stage
X_train_stage2_raw = education_train_processed_outliers_handled.drop('vacation', axis=1)
y_train_stage2 = education_train_processed_outliers_handled['vacation']
X_test_stage2_raw = education_test_processed_outliers_handled.drop('vacation', axis=1)
y_test_stage2 = education_test_processed_outliers_handled['vacation']

# Apply One-Hot Encoding
ohe_stage2 = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_stage2_encoded = ohe_stage2.fit_transform(X_train_stage2_raw[categorical_features_for_ohe])
X_train_stage2_encoded_df = pd.DataFrame(X_train_stage2_encoded, columns=ohe_stage2.get_feature_names_out(categorical_features_for_ohe), index=X_train_stage2_raw.index)
X_train_stage2_processed = X_train_stage2_raw.drop(columns=categorical_features_for_ohe)
X_train_stage2_processed = pd.concat([X_train_stage2_processed.reset_index(drop=True), X_train_stage2_encoded_df.reset_index(drop=True)], axis=1)

X_test_stage2_encoded = ohe_stage2.transform(X_test_stage2_raw[categorical_features_for_ohe])
X_test_stage2_encoded_df = pd.DataFrame(X_test_stage2_encoded, columns=ohe_stage2.get_feature_names_out(categorical_features_for_ohe), index=X_test_stage2_raw.index)
X_test_stage2_processed = X_test_stage2_raw.drop(columns=categorical_features_for_ohe)
X_test_stage2_processed = pd.concat([X_test_stage2_processed.reset_index(drop=True), X_test_stage2_encoded_df.reset_index(drop=True)], axis=1)

# Train and evaluate Decision Tree Classifier
dt_classifier_stage2 = DecisionTreeClassifier()
dt_classifier_stage2.fit(X_train_stage2_processed, y_train_stage2)
y_pred_stage2 = dt_classifier_stage2.predict(X_test_stage2_processed)

print("--- Results for Outlier Handling and Numerical Imputation ---")
print(f"Accuracy: {accuracy_score(y_test_stage2, y_pred_stage2):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_stage2, y_pred_stage2))

### Model Evaluation on Fully Preprocessed Dataset (Imputation, Outlier Handling, Scaling)

In [ ]:
# The variables X_train, y_train, X_test, y_test already correspond to the final preprocessed data.
# The model has already been trained and evaluated in previous steps.
# We will just re-print the results for clarity here.

# Ensure X_train_raw, X_test_raw are correctly loaded from education_train_processed_final
# Split the education_train_processed_final into training (80%) and validation (20%) sets
X_full_processed = education_train_processed_final.drop('vacation', axis=1)
y_full_processed = education_train_processed_final['vacation']

X_train_stage3_raw, X_test_stage3_raw, y_train_stage3, y_test_stage3 = train_test_split(
    X_full_processed, y_full_processed, test_size=0.2, random_state=42, stratify=y_full_processed
)

ohe_stage3 = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_stage3_encoded = ohe_stage3.fit_transform(X_train_stage3_raw[categorical_features_for_ohe])
X_train_stage3_encoded_df = pd.DataFrame(X_train_stage3_encoded, columns=ohe_stage3.get_feature_names_out(categorical_features_for_ohe), index=X_train_stage3_raw.index)
X_train_stage3_processed = X_train_stage3_raw.drop(columns=categorical_features_for_ohe)
X_train_stage3_processed = pd.concat([X_train_stage3_processed.reset_index(drop=True), X_train_stage3_encoded_df.reset_index(drop=True)], axis=1)

X_test_stage3_encoded = ohe_stage3.transform(X_test_stage3_raw[categorical_features_for_ohe])
X_test_stage3_encoded_df = pd.DataFrame(X_test_stage3_encoded, columns=ohe_stage3.get_feature_names_out(categorical_features_for_ohe), index=X_test_stage3_raw.index)
X_test_stage3_processed = X_test_stage3_raw.drop(columns=categorical_features_for_ohe)
X_test_stage3_processed = pd.concat([X_test_stage3_processed.reset_index(drop=True), X_test_stage3_encoded_df.reset_index(drop=True)], axis=1)

# Train and evaluate Decision Tree Classifier
dt_classifier_stage3 = DecisionTreeClassifier()
dt_classifier_stage3.fit(X_train_stage3_processed, y_train_stage3)
y_pred_stage3 = dt_classifier_stage3.predict(X_test_stage3_processed)

print("--- Results for Fully Preprocessed Dataset (using 80/20 train/validation split from original training data) ---")
print(f"Accuracy: {accuracy_score(y_test_stage3, y_pred_stage3):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_stage3, y_pred_stage3))

### Sumar al performanței modelului pe diferite etape de preprocesare

In [ ]:
results = {
    'Etapa de Preprocesare': [
        'Imputare Categorică',
        'Outlier Handling și Imputare Numerică',
        'Complet Preprocesat (Imputare, Outlier Handling, Scalare)'
    ],
    'Acuratețe': [
        accuracy_score(y_test_stage1, y_pred_stage1), # Results from 92dbdd76 (Categorical Imputation Only)
        accuracy_score(y_test_stage2, y_pred_stage2), # Results from 2f27ce89 (Outlier Handling and Numerical Imputation)
        accuracy_score(y_test_stage3, y_pred_stage3)  # Results from e3067e54 (Fully Preprocessed)
    ],
    'F1-Score (weighted avg)': [
        classification_report(y_test_stage1, y_pred_stage1, output_dict=True)['weighted avg']['f1-score'],
        classification_report(y_test_stage2, y_pred_stage2, output_dict=True)['weighted avg']['f1-score'],
        classification_report(y_test_stage3, y_pred_stage3, output_dict=True)['weighted avg']['f1-score']
    ]
}

results_df = pd.DataFrame(results)
display(results_df)

### In continuare vom folosi doar setul de date cu imputare categorica avand in vedere ca aduce cea mai buna performanta (chiar daca diferenta este foarte mica)

### Evaluarea modelului cu parametri impliciți (fără tuning)

In [ ]:
print("--- Rulare cu parametri impliciți ---")
dt_classifier_default = DecisionTreeClassifier()
dt_classifier_default.fit(X_train_stage1_processed, y_train_stage1)
y_pred_default = dt_classifier_default.predict(X_test_stage1_processed)

accuracy_default = accuracy_score(y_test_stage1, y_pred_default)
f1_weighted_default = classification_report(y_test_stage1, y_pred_default, output_dict=True)['weighted avg']['f1-score']

# Creăm un DataFrame concis cu rezultatele modelului implicit
default_results = {
    'Parametru': ['Implicit'],
    'Acuratețe': [accuracy_default],
    'F1-Score (weighted avg)': [f1_weighted_default]
}

default_results_df = pd.DataFrame(default_results)
print(f"Accuracy: {accuracy_default:.4f}")
print(f"F1-Score (weighted avg): {f1_weighted_default:.4f}")

display(default_results_df)

### Evaluarea modelului cu diferite valori pentru hiperparametrul `min_samples_leaf`

In [ ]:
min_samples_leaf_values = [2, 5, 10, 12, 15, 20, 25, 30, 40, 50, 100]
results_min_samples_leaf = []

for min_leaf in min_samples_leaf_values:
    print(f"\n--- Rulare cu min_samples_leaf = {min_leaf} ---")
    dt_classifier_tuned = DecisionTreeClassifier(min_samples_leaf=min_leaf)
    dt_classifier_tuned.fit(X_train_stage3_processed, y_train_stage3)
    y_pred_tuned = dt_classifier_tuned.predict(X_test_stage3_processed)

    accuracy = accuracy_score(y_test_stage3, y_pred_tuned)
    f1_weighted = classification_report(y_test_stage3, y_pred_tuned, output_dict=True)['weighted avg']['f1-score']

    print(f"Accuracy: {accuracy:.4f}")
    print("Classification Report:")
    print(classification_report(y_test_stage3, y_pred_tuned))

    results_min_samples_leaf.append({
        'min_samples_leaf': min_leaf,
        'Acuratețe': accuracy,
        'F1-Score (weighted avg)': f1_weighted
    })

results_min_samples_leaf_df = pd.DataFrame(results_min_samples_leaf)
print("\nSumar al performanței cu diferite min_samples_leaf:")
display(results_min_samples_leaf_df)


### Cel mai bun rezultat il obtin pentru min_samples_leaf = 40 deci cu el vom merge mai departe

### Evaluarea modelului cu diferite valori pentru hiperparametrul `min_samples_split`

In [ ]:
min_samples_split_values = [60, 70, 100, 120, 150, 200]
results_min_samples_split = []

for min_split in min_samples_split_values:
    print(f"\n--- Rulare cu min_samples_split = {min_split} ---")
    dt_classifier_tuned = DecisionTreeClassifier(min_samples_split=min_split)
    dt_classifier_tuned.fit(X_train_stage3_processed, y_train_stage3)
    y_pred_tuned = dt_classifier_tuned.predict(X_test_stage3_processed)

    accuracy = accuracy_score(y_test_stage3, y_pred_tuned)
    f1_weighted = classification_report(y_test_stage3, y_pred_tuned, output_dict=True)['weighted avg']['f1-score']

    print(f"Accuracy: {accuracy:.4f}")
    print("Classification Report:")
    print(classification_report(y_test_stage3, y_pred_tuned))

    results_min_samples_split.append({
        'min_samples_split': min_split,
        'Acuratețe': accuracy,
        'F1-Score (weighted avg)': f1_weighted
    })

results_min_samples_split_df = pd.DataFrame(results_min_samples_split)
print("\nSumar al performanței cu diferite min_samples_split:")
display(results_min_samples_split_df)

### Voi merge mai departe cu min_samples_split = 150

### Evaluarea modelului cu diferite valori pentru hiperparametrul `max_depth`

In [ ]:
max_depth_values = [5, 8, 10, 15, 20]
results_max_depth = []

for max_depth in max_depth_values:
    print(f"\n--- Rulare cu max_depth = {max_depth} ---")
    dt_classifier_tuned = DecisionTreeClassifier(max_depth=max_depth)
    dt_classifier_tuned.fit(X_train_stage1_processed, y_train_stage1)
    y_pred_tuned = dt_classifier_tuned.predict(X_test_stage1_processed)

    accuracy = accuracy_score(y_test_stage1, y_pred_tuned)
    f1_weighted = classification_report(y_test_stage1, y_pred_tuned, output_dict=True)['weighted avg']['f1-score']

    print(f"Accuracy: {accuracy:.4f}")
    print("Classification Report:")
    print(classification_report(y_test_stage1, y_pred_tuned))

    results_max_depth.append({
        'max_depth': max_depth,
        'Acuratețe': accuracy,
        'F1-Score (weighted avg)': f1_weighted
    })

results_max_depth_df = pd.DataFrame(results_max_depth)
print("\nSumar al performanței cu diferite max_depth:")
display(results_max_depth_df)

### Voi merge mai departe cu max_depth = 10

### Evaluarea modelului cu `class_weight = 'balanced'`

In [ ]:
print("--- Rulare cu class_weight = 'balanced' ---")
dt_classifier_balanced = DecisionTreeClassifier(class_weight='balanced')
dt_classifier_balanced.fit(X_train_stage3_processed, y_train_stage3)
y_pred_balanced = dt_classifier_balanced.predict(X_test_stage3_processed)

accuracy_balanced = accuracy_score(y_test_stage3, y_pred_balanced)
f1_weighted_balanced = classification_report(y_test_stage3, y_pred_balanced, output_dict=True)['weighted avg']['f1-score']

print(f"Accuracy: {accuracy_balanced:.4f}")
print("Classification Report:")
print(classification_report(y_test_stage3, y_pred_balanced))

balanced_results = {
    'Parametru': ['class_weight = balanced'],
    'Acuratețe': [accuracy_balanced],
    'F1-Score (weighted avg)': [f1_weighted_balanced]
}

balanced_results_df = pd.DataFrame(balanced_results)
display(balanced_results_df)

### Reechilibrarea claselor cu SMOTE și reevaluarea modelului

In [ ]:
# Importăm SMOTE
from imblearn.over_sampling import SMOTE
from collections import Counter

print("--- Rulare cu SMOTE ---")

# Datele noastre sunt deja codificate one-hot (X_train_stage3_processed, y_train_stage3)

# Verificăm distribuția claselor înainte de SMOTE
print(f"Distribuția claselor înainte de SMOTE: {Counter(y_train_stage3)}")

# Inițializăm SMOTE
smote = SMOTE(random_state=42)

# Aplicăm SMOTE pe setul de antrenament
X_train_smote, y_train_smote = smote.fit_resample(X_train_stage3_processed, y_train_stage3)

# Verificăm distribuția claselor după SMOTE
print(f"Distribuția claselor după SMOTE: {Counter(y_train_smote)}")

# Antrenăm un nou Decision Tree Classifier pe datele reechilibrate
dt_classifier_smote = DecisionTreeClassifier(random_state=42)
dt_classifier_smote.fit(X_train_smote, y_train_smote)

# Facem predicții pe setul de testare (neafectat de SMOTE)
y_pred_smote = dt_classifier_smote.predict(X_test_stage3_processed)

# Evaluăm performanța
accuracy_smote = accuracy_score(y_test_stage3, y_pred_smote)
f1_weighted_smote = classification_report(y_test_stage3, y_pred_smote, output_dict=True)['weighted avg']['f1-score']

print(f"\nAccuracy după SMOTE: {accuracy_smote:.4f}")
print("\nClassification Report după SMOTE:")
print(classification_report(y_test_stage3, y_pred_smote))

smote_results = {
    'Parametru': ['SMOTE aplicat'],
    'Acuratețe': [accuracy_smote],
    'F1-Score (weighted avg)': [f1_weighted_smote]
}

smote_results_df = pd.DataFrame(smote_results)
display(smote_results_df)

### Evaluarea modelului cu `min_samples_split = 150` și `min_samples_leaf = 40`

In [ ]:
print("--- Rulare cu min_samples_split = 150 și min_samples_leaf = 40 ---")

dt_classifier_combined_tuned = DecisionTreeClassifier(min_samples_split=150, min_samples_leaf=40, random_state=42)
dt_classifier_combined_tuned.fit(X_train_stage1_processed, y_train_stage1)
y_pred_combined_tuned = dt_classifier_combined_tuned.predict(X_test_stage1_processed)

accuracy_combined_tuned = accuracy_score(y_test_stage1, y_pred_combined_tuned)
f1_weighted_combined_tuned = classification_report(y_test_stage1, y_pred_combined_tuned, output_dict=True)['weighted avg']['f1-score']

print(f"Accuracy: {accuracy_combined_tuned:.4f}")
print("Classification Report:")
print(classification_report(y_test_stage1, y_pred_combined_tuned))

combined_tuned_results = {
    'Parametru': ['min_samples_split=150, min_samples_leaf=40'],
    'Acuratețe': [accuracy_combined_tuned],
    'F1-Score (weighted avg)': [f1_weighted_combined_tuned]
}

combined_tuned_results_df = pd.DataFrame(combined_tuned_results)
display(combined_tuned_results_df)

### Evaluarea modelului cu `min_samples_split = 150`, `min_samples_leaf = 40` și `max_depth = 10`

In [ ]:
print("--- Rulare cu min_samples_split = 150, min_samples_leaf = 40 și max_depth = 10 ---")

dt_classifier_final_tuned = DecisionTreeClassifier(min_samples_split=150, min_samples_leaf=40, max_depth=10, random_state=42)
dt_classifier_final_tuned.fit(X_train_stage1_processed, y_train_stage1)
y_pred_final_tuned = dt_classifier_final_tuned.predict(X_test_stage1_processed)

accuracy_final_tuned = accuracy_score(y_test_stage1, y_pred_final_tuned)
f1_weighted_final_tuned = classification_report(y_test_stage1, y_pred_final_tuned, output_dict=True)['weighted avg']['f1-score']

print(f"Accuracy: {accuracy_final_tuned:.4f}")
print("Classification Report:")
print(classification_report(y_test_stage1, y_pred_final_tuned))

final_tuned_results = {
    'Parametru': ['min_samples_split=150, min_samples_leaf=40, max_depth=10'],
    'Acuratețe': [accuracy_final_tuned],
    'F1-Score (weighted avg)': [f1_weighted_final_tuned]
}

final_tuned_results_df = pd.DataFrame(final_tuned_results)
display(final_tuned_results_df)

### Observ ca max_depth = 10 imi aduce o mica imbunatatire la scorurile f1 ale claselor greu de distins (small si medium), dar cu pretul unui scor F1 mediu mai mic si o acuratete mai mica

### Evaluarea modelului cu `min_samples_split = 150`, `min_samples_leaf = 40`, `max_depth = 10`, `class_weight = 'balanced'` și SMOTE

In [ ]:
print("--- Rulare cu min_samples_split = 150, min_samples_leaf = 40, max_depth = 10, class_weight = 'balanced' și SMOTE ---")

# Apply SMOTE to the training data (from stage 1 as per user's last instruction)
smote_combined = SMOTE(random_state=42)
X_train_smote_combined, y_train_smote_combined = smote_combined.fit_resample(X_train_stage1_processed, y_train_stage1)

# Train Decision Tree Classifier with combined parameters on SMOTE-resampled data
dt_classifier_combined = DecisionTreeClassifier(min_samples_split=150, min_samples_leaf=40, max_depth=10, class_weight='balanced', random_state=42)
dt_classifier_combined.fit(X_train_smote_combined, y_train_smote_combined)
y_pred_combined = dt_classifier_combined.predict(X_test_stage1_processed)

accuracy_combined = accuracy_score(y_test_stage1, y_pred_combined)
f1_weighted_combined = classification_report(y_test_stage1, y_pred_combined, output_dict=True)['weighted avg']['f1-score']

print(f"Accuracy: {accuracy_combined:.4f}")
print("Classification Report:")
print(classification_report(y_test_stage1, y_pred_combined))

combined_results = {
    'Parametru': ["SMOTE + class_weight='balanced' + min_samples_split=150 + min_samples_leaf=40 + max_depth=10"],
    'Acuratețe': [accuracy_combined],
    'F1-Score (weighted avg)': [f1_weighted_combined]
}

combined_results_df = pd.DataFrame(combined_results)
display(combined_results_df)

SMOTE si class_weight imi cresc si ele scorul f1 pt clasa Small insa pierd pe partea de acuratete si scor F1 mediu

### Evaluarea modelului cu `min_samples_split = 150`, `min_samples_leaf = 40` și fără coloana `industry`

In [ ]:
# Identify columns related to 'industry' from the one-hot encoded features
industry_cols_to_drop = [col for col in X_train_stage1_processed.columns if col.startswith('industry_')]

# Create copies of the processed dataframes and drop the identified columns
X_train_no_industry = X_train_stage1_processed.drop(columns=industry_cols_to_drop)
X_test_no_industry = X_test_stage1_processed.drop(columns=industry_cols_to_drop)

print(f"Original number of features: {X_train_stage1_processed.shape[1]}")
print(f"Number of features after dropping 'industry': {X_train_no_industry.shape[1]}")

# Train Decision Tree Classifier with specified hyperparameters on the modified data
dt_classifier_no_industry = DecisionTreeClassifier(min_samples_split=150, min_samples_leaf=40, random_state=42)
dt_classifier_no_industry.fit(X_train_no_industry, y_train_stage1)
y_pred_no_industry = dt_classifier_no_industry.predict(X_test_no_industry)

accuracy_no_industry = accuracy_score(y_test_stage1, y_pred_no_industry)
f1_weighted_no_industry = classification_report(y_test_stage1, y_pred_no_industry, output_dict=True)['weighted avg']['f1-score']

print(f"\n--- Rulare cu min_samples_split = 150, min_samples_leaf = 40 și fără coloana industry ---")
print(f"Accuracy: {accuracy_no_industry:.4f}")
print("Classification Report:")
print(classification_report(y_test_stage1, y_pred_no_industry))

no_industry_results = {
    'Parametru': ['min_samples_split=150, min_samples_leaf=40, fără industry'],
    'Acuratețe': [accuracy_no_industry],
    'F1-Score (weighted avg)': [f1_weighted_no_industry]
}

no_industry_results_df = pd.DataFrame(no_industry_results)
display(no_industry_results_df)

Am observat din matricea de corelatie faprul ca atributul industry nu este aproape deloc relevant pentru prezicerea clasei 'Vacation' asa ca l-am eliminat

### Evaluarea modelului cu `min_samples_split = 150`, `min_samples_leaf = 40` și fără coloanele `industry` și `total_days_worked`

In [ ]:
# Identify columns related to 'industry' and 'total_days_worked' from the one-hot encoded features
industry_cols_to_drop = [col for col in X_train_stage1_processed.columns if col.startswith('industry_')]
columns_to_drop = industry_cols_to_drop + ['total_days_worked'] + ['salary']

# Create copies of the processed dataframes and drop the identified columns
X_train_reduced = X_train_stage1_processed.drop(columns=columns_to_drop)
X_test_reduced = X_test_stage1_processed.drop(columns=columns_to_drop)

print(f"Original number of features: {X_train_stage1_processed.shape[1]}")
print(f"Number of features after dropping 'industry' and 'total_days_worked': {X_train_reduced.shape[1]}")

# Train Decision Tree Classifier with specified hyperparameters on the modified data
dt_classifier_reduced = DecisionTreeClassifier(min_samples_split=150, min_samples_leaf=40, random_state=42)
dt_classifier_reduced.fit(X_train_reduced, y_train_stage1)
y_pred_reduced = dt_classifier_reduced.predict(X_test_reduced)

accuracy_reduced = accuracy_score(y_test_stage1, y_pred_reduced)
f1_weighted_reduced = classification_report(y_test_stage1, y_pred_reduced, output_dict=True)['weighted avg']['f1-score']

print(f"\n--- Rulare cu min_samples_split = 150, min_samples_leaf = 40 și fără coloanele industry și total_days_worked ---")
print(f"Accuracy: {accuracy_reduced:.4f}")
print("Classification Report:")
print(classification_report(y_test_stage1, y_pred_reduced))

reduced_results = {
    'Parametru': ['min_samples_split=150, min_samples_leaf=40, fără industry și total_days_worked'],
    'Acuratețe': [accuracy_reduced],
    'F1-Score (weighted avg)': [f1_weighted_reduced]
}

reduced_results_df = pd.DataFrame(reduced_results)
display(reduced_results_df)

Elimin si total_days_worked, pentru ca este puternic corelat cu experience_years

### Sumar al performanței modelului pe diferite etape de preprocesare și modele

In [ ]:
all_results_dfs = [
    results_df.rename(columns={'Etapa de Preprocesare': 'Parametru'}),
    default_results_df,
    results_min_samples_leaf_df.copy().rename(columns={'min_samples_samples_leaf': 'Parametru'}),
    results_min_samples_split_df.copy().rename(columns={'min_samples_split': 'Parametru'}),
    results_max_depth_df.copy().rename(columns={'max_depth': 'Parametru'}),
    balanced_results_df,
    smote_results_df,
    combined_tuned_results_df,
    final_tuned_results_df,
    no_industry_results_df,
    reduced_results_df
]

# Concatenate all results into a single DataFrame
consolidated_results_df = pd.concat(all_results_dfs, ignore_index=True)

# Sort by F1-Score (weighted avg) in descending order
consolidated_results_df_sorted = consolidated_results_df.sort_values(by='F1-Score (weighted avg)', ascending=False).reset_index(drop=True)

display(consolidated_results_df_sorted)

### Cel mai bun rezultat l-am obtinut pentru min_samples_split = 150, min_samples_leaf = 40 si fara coloana industry, pe setul de date pe care am facut doar imputarea atributelor categorice

### Regresie

### Pregătirea Datelor pentru Regresia Salariului

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression

# Use the dataset from the categorical imputation stage, as it was chosen for classification tasks.
X_train_reg_raw = education_train_processed_categorical_imputed.drop(['salary', 'vacation'], axis=1)
y_train_reg = education_train_processed_categorical_imputed['salary']
X_test_reg_raw = education_test_processed_categorical_imputed.drop(['salary', 'vacation'], axis=1)
y_test_reg = education_test_processed_categorical_imputed['salary']

# Identify categorical and numerical features for this regression task
# 'categoric_ordinal_attributes' and 'numerical_continuous_attributes' are defined earlier in the notebook.
categorical_features_reg = [col for col in categoric_ordinal_attributes if col != 'vacation']
numerical_features_reg = numerical_continuous_attributes.copy()
if 'salary' in numerical_features_reg:
    numerical_features_reg.remove('salary') # Remove salary from numerical features, as it's the target

# Apply One-Hot Encoding to categorical features
ohe_reg = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_reg_encoded = ohe_reg.fit_transform(X_train_reg_raw[categorical_features_reg])
X_train_reg_encoded_df = pd.DataFrame(X_train_reg_encoded, columns=ohe_reg.get_feature_names_out(categorical_features_reg), index=X_train_reg_raw.index)

X_test_reg_encoded = ohe_reg.transform(X_test_reg_raw[categorical_features_reg])
X_test_reg_encoded_df = pd.DataFrame(X_test_reg_encoded, columns=ohe_reg.get_feature_names_out(categorical_features_reg), index=X_test_reg_raw.index)

# Scale numerical features
scaler_reg = StandardScaler()
X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg_raw[numerical_features_reg])
X_train_reg_scaled_df = pd.DataFrame(X_train_reg_scaled, columns=numerical_features_reg, index=X_train_reg_raw.index)

X_test_reg_scaled = scaler_reg.transform(X_test_reg_raw[numerical_features_reg])
X_test_reg_scaled_df = pd.DataFrame(X_test_reg_scaled, columns=numerical_features_reg, index=X_test_reg_raw.index)

# Combine processed features
X_train_reg_processed = pd.concat([X_train_reg_scaled_df.reset_index(drop=True), X_train_reg_encoded_df.reset_index(drop=True)], axis=1)
X_test_reg_processed = pd.concat([X_test_reg_scaled_df.reset_index(drop=True), X_test_reg_encoded_df.reset_index(drop=True)], axis=1)

print("Data preprocesată pentru regresie.")

### Regresie Liniară pentru Salariu

In [ ]:
print("--- Antrenare model de Regresie Liniară ---")

# Train a LinearRegression model
linear_regressor = LinearRegression()
linear_regressor.fit(X_train_reg_processed, y_train_reg)

# Make predictions on the test set
y_pred_linear_reg = linear_regressor.predict(X_test_reg_processed)

# Evaluate the model
mae_linear = mean_absolute_error(y_test_reg, y_pred_linear_reg)
mse_linear = mean_squared_error(y_test_reg, y_pred_linear_reg)
rmse_linear = np.sqrt(mse_linear)
r2_linear = r2_score(y_test_reg, y_pred_linear_reg)

print("--- Rezultate Regresie Liniară Salariu ---")
print(f"Mean Absolute Error (MAE): {mae_linear:.2f}")
print(f"Mean Squared Error (MSE): {mse_linear:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse_linear:.2f}")
print(f"R-squared (R2): {r2_linear:.4f}")

Modelul de regresie liniară a fost antrenat pentru a prezice `salary`.

**Interpretarea rezultatelor (Linear Regression):**
- **Mean Absolute Error (MAE)**: Eroarea medie absolută. O valoare mai mică indică predicții mai precise.
- **Mean Squared Error (MSE)**: Pătratul erorii medii. Penalizează erorile mari mai mult decât MAE.
- **Root Mean Squared Error (RMSE)**: Rădăcina pătrată a MSE, ușor de interpretat deoarece este în aceleași unități cu variabila țintă.
- **R-squared (R2)**: Coeficientul de determinare. Indică proporția varianței din variabila dependentă (`salary`) care este explicată de variabilele independente din model. O valoare mai apropiată de 1 sugerează o potrivire mai bună a modelului.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

plt.figure(figsize=(10, 8))
plt.scatter(y_test_reg, y_pred_linear_reg, alpha=0.3)

# Plot the ideal line (y=x)
plt.plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], 'r--', lw=2, label='Ideal Prediction Line (y=x)')

# Fit a simple linear model to the actual vs predicted points to get the prediction trend line
reg_line_model = LinearRegression()
reg_line_model.fit(y_test_reg.values.reshape(-1, 1), y_pred_linear_reg)

# Generate points for the prediction trend line
x_vals = np.array([y_test_reg.min(), y_test_reg.max()]).reshape(-1, 1)
y_vals_reg_line = reg_line_model.predict(x_vals)

# Plot the prediction trend line (in orange)
plt.plot(x_vals, y_vals_reg_line, color='orange', lw=2, label='Model Prediction Trend Line')

plt.xlabel('Actual Salary')
plt.ylabel('Predicted Salary')
plt.title('Actual vs. Predicted Salary with Ideal and Model Trend Lines (Linear Regression)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend() # Show legend to distinguish lines
plt.show()

### Regresie Ridge pentru Salariu

Pt alpha = 1

In [ ]:
from sklearn.linear_model import Ridge

print("--- Antrenare model de Regresie Ridge ---")

# Train a Ridge Regression model
# You can experiment with the 'alpha' parameter (regularization strength)
ridge_regressor = Ridge(alpha=1.0) # Default alpha, can be tuned
ridge_regressor.fit(X_train_reg_processed, y_train_reg)

# Make predictions on the test set
y_pred_ridge_reg = ridge_regressor.predict(X_test_reg_processed)

# Evaluate the model
mae_ridge = mean_absolute_error(y_test_reg, y_pred_ridge_reg)
mse_ridge = mean_squared_error(y_test_reg, y_pred_ridge_reg)
rmse_ridge = np.sqrt(mse_ridge)
r2_ridge = r2_score(y_test_reg, y_pred_ridge_reg)

print("--- Rezultate Regresie Ridge Salariu ---")
print(f"Mean Absolute Error (MAE): {mae_ridge:.2f}")
print(f"Mean Squared Error (MSE): {mse_ridge:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse_ridge:.2f}")
print(f"R-squared (R2): {r2_ridge:.4f}")

In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(y_test_reg, y_pred_ridge_reg, alpha=0.3)

# Plot the ideal line (y=x)
plt.plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], 'r--', lw=2, label='Ideal Prediction Line (y=x)')

# Fit a simple linear model to the actual vs predicted points to get the prediction trend line
reg_line_model_ridge = LinearRegression()
reg_line_model_ridge.fit(y_test_reg.values.reshape(-1, 1), y_pred_ridge_reg)

# Generate points for the prediction trend line
x_vals_ridge = np.array([y_test_reg.min(), y_test_reg.max()]).reshape(-1, 1)
y_vals_reg_line_ridge = reg_line_model_ridge.predict(x_vals_ridge)

# Plot the prediction trend line (in orange)
plt.plot(x_vals_ridge, y_vals_reg_line_ridge, color='orange', lw=2, label='Model Prediction Trend Line')

plt.xlabel('Actual Salary')
plt.ylabel('Predicted Salary')
plt.title('Actual vs. Predicted Salary with Ideal and Model Trend Lines (Ridge Regression)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.show()

### Regresie Ridge pentru Salariu cu alpha = 1.25

In [ ]:
from sklearn.linear_model import Ridge

print("--- Antrenare model de Regresie Ridge cu alpha = 1.25 ---")

ridge_regressor_1_25 = Ridge(alpha=1.25)
ridge_regressor_1_25.fit(X_train_reg_processed, y_train_reg)

y_pred_ridge_reg_1_25 = ridge_regressor_1_25.predict(X_test_reg_processed)

mae_ridge_1_25 = mean_absolute_error(y_test_reg, y_pred_ridge_reg_1_25)
mse_ridge_1_25 = mean_squared_error(y_test_reg, y_pred_ridge_reg_1_25)
rmse_ridge_1_25 = np.sqrt(mse_ridge_1_25)
r2_ridge_1_25 = r2_score(y_test_reg, y_pred_ridge_reg_1_25)

print("--- Rezultate Regresie Ridge Salariu cu alpha = 1.25 ---")
print(f"Mean Absolute Error (MAE): {mae_ridge_1_25:.2f}")
print(f"Mean Squared Error (MSE): {mse_ridge_1_25:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse_ridge_1_25:.2f}")
print(f"R-squared (R2): {r2_ridge_1_25:.4f}")

In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(y_test_reg, y_pred_ridge_reg_1_25, alpha=0.3)

plt.plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], 'r--', lw=2, label='Ideal Prediction Line (y=x)')

reg_line_model_ridge_1_25 = LinearRegression()
reg_line_model_ridge_1_25.fit(y_test_reg.values.reshape(-1, 1), y_pred_ridge_reg_1_25)

x_vals_ridge_1_25 = np.array([y_test_reg.min(), y_test_reg.max()]).reshape(-1, 1)
y_vals_reg_line_ridge_1_25 = reg_line_model_ridge_1_25.predict(x_vals_ridge_1_25)

plt.plot(x_vals_ridge_1_25, y_vals_reg_line_ridge_1_25, color='orange', lw=2, label='Model Prediction Trend Line')

plt.xlabel('Actual Salary')
plt.ylabel('Predicted Salary')
plt.title('Actual vs. Predicted Salary with Ideal and Model Trend Lines (Ridge Regression, alpha=1.25)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.show()

### Regresie Ridge pentru Salariu cu alpha = 1.5

In [ ]:
from sklearn.linear_model import Ridge

print("--- Antrenare model de Regresie Ridge cu alpha = 1.5 ---")

ridge_regressor_1_5 = Ridge(alpha=1.5)
ridge_regressor_1_5.fit(X_train_reg_processed, y_train_reg)

y_pred_ridge_reg_1_5 = ridge_regressor_1_5.predict(X_test_reg_processed)

mae_ridge_1_5 = mean_absolute_error(y_test_reg, y_pred_ridge_reg_1_5)
mse_ridge_1_5 = mean_squared_error(y_test_reg, y_pred_ridge_reg_1_5)
rmse_ridge_1_5 = np.sqrt(mse_ridge_1_5)
r2_ridge_1_5 = r2_score(y_test_reg, y_pred_ridge_reg_1_5)

print("--- Rezultate Regresie Ridge Salariu cu alpha = 1.5 ---")
print(f"Mean Absolute Error (MAE): {mae_ridge_1_5:.2f}")
print(f"Mean Squared Error (MSE): {mse_ridge_1_5:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse_ridge_1_5:.2f}")
print(f"R-squared (R2): {r2_ridge_1_5:.4f}")

In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(y_test_reg, y_pred_ridge_reg_1_5, alpha=0.3)

plt.plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], 'r--', lw=2, label='Ideal Prediction Line (y=x)')

reg_line_model_ridge_1_5 = LinearRegression()
reg_line_model_ridge_1_5.fit(y_test_reg.values.reshape(-1, 1), y_pred_ridge_reg_1_5)

x_vals_ridge_1_5 = np.array([y_test_reg.min(), y_test_reg.max()]).reshape(-1, 1)
y_vals_reg_line_ridge_1_5 = reg_line_model_ridge_1_5.predict(x_vals_ridge_1_5)

plt.plot(x_vals_ridge_1_5, y_vals_reg_line_ridge_1_5, color='orange', lw=2, label='Model Prediction Trend Line')

plt.xlabel('Actual Salary')
plt.ylabel('Predicted Salary')
plt.title('Actual vs. Predicted Salary with Ideal and Model Trend Lines (Ridge Regression, alpha=1.5)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.show()

### Regresie Ridge pentru Salariu cu alpha = 1.75

In [ ]:
from sklearn.linear_model import Ridge

print("--- Antrenare model de Regresie Ridge cu alpha = 1.75 ---")

ridge_regressor_1_75 = Ridge(alpha=1.75)
ridge_regressor_1_75.fit(X_train_reg_processed, y_train_reg)

y_pred_ridge_reg_1_75 = ridge_regressor_1_75.predict(X_test_reg_processed)

mae_ridge_1_75 = mean_absolute_error(y_test_reg, y_pred_ridge_reg_1_75)
mse_ridge_1_75 = mean_squared_error(y_test_reg, y_pred_ridge_reg_1_75)
rmse_ridge_1_75 = np.sqrt(mse_ridge_1_75)
r2_ridge_1_75 = r2_score(y_test_reg, y_pred_ridge_reg_1_75)

print("--- Rezultate Regresie Ridge Salariu cu alpha = 1.75 ---")
print(f"Mean Absolute Error (MAE): {mae_ridge_1_75:.2f}")
print(f"Mean Squared Error (MSE): {mse_ridge_1_75:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse_ridge_1_75:.2f}")
print(f"R-squared (R2): {r2_ridge_1_75:.4f}")

In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(y_test_reg, y_pred_ridge_reg_1_75, alpha=0.3)

plt.plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], 'r--', lw=2, label='Ideal Prediction Line (y=x)')

reg_line_model_ridge_1_75 = LinearRegression()
reg_line_model_ridge_1_75.fit(y_test_reg.values.reshape(-1, 1), y_pred_ridge_reg_1_75)

x_vals_ridge_1_75 = np.array([y_test_reg.min(), y_test_reg.max()]).reshape(-1, 1)
y_vals_reg_line_ridge_1_75 = reg_line_model_ridge_1_75.predict(x_vals_ridge_1_75)

plt.plot(x_vals_ridge_1_75, y_vals_reg_line_ridge_1_75, color='orange', lw=2, label='Model Prediction Trend Line')

plt.xlabel('Actual Salary')
plt.ylabel('Predicted Salary')
plt.title('Actual vs. Predicted Salary with Ideal and Model Trend Lines (Ridge Regression, alpha=1.75)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.show()

### Compararea performanței modelelor de regresie

In [ ]:
regression_results = {
    'Model': [
        'Linear Regression',
        'Ridge Regression (alpha=1.0)',
        'Ridge Regression (alpha=1.25)',
        'Ridge Regression (alpha=1.5)',
        'Ridge Regression (alpha=1.75)'
    ],
    'MAE': [
        mae_linear,
        mae_ridge,
        mae_ridge_1_25,
        mae_ridge_1_5,
        mae_ridge_1_75
    ],
    'MSE': [
        mse_linear,
        mse_ridge,
        mse_ridge_1_25,
        mse_ridge_1_5,
        mse_ridge_1_75
    ],
    'RMSE': [
        rmse_linear,
        rmse_ridge,
        rmse_ridge_1_25,
        rmse_ridge_1_5,
        rmse_ridge_1_75
    ],
    'R-squared': [
        r2_linear,
        r2_ridge,
        r2_ridge_1_25,
        r2_ridge_1_5,
        r2_ridge_1_75
    ]
}

regression_results_df = pd.DataFrame(regression_results)

# Sort by RMSE to find the best performing model (lower RMSE is better)
regression_results_df_sorted = regression_results_df.sort_values(by='RMSE').reset_index(drop=True)

print("Sumarul performanței modelelor de regresie:")
display(regression_results_df_sorted)

Din tabelul de mai sus, observăm că toate modelele de regresie (Linear Regression și variantele Ridge Regression cu diferite `alpha`) au performanțe extrem de similare. Diferențele în RMSE și MAE sunt minime, iar R-squared rămâne constant la `0.9600`.

Aceasta sugerează că pentru acest set de date, regularizarea (prin Ridge) nu aduce îmbunătățiri semnificative față de regresia liniară simplă. Este posibil ca setul de date să nu prezinte o multicoliniaritate puternică sau overfitting, sau că gama de valori `alpha` testate este prea restrânsă pentru a observa un impact mai mare.

Prin urmare, în acest caz, nu există un `alpha` care să se distingă semnificativ ca fiind "mai bun" în gama explorată, iar modelul de regresie liniară simplă este la fel de eficient.

### Regresie Polinomială pentru Salariu

Chiar dacă modelele liniare au arătat deja o performanță excelentă, vom explora regresia polinomială pentru a vedea dacă o relație non-liniară, mai complexă, poate aduce îmbunătățiri. Vom testa cu diferite grade polinomiale.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

print("--- Antrenare model de Regresie Polinomială (grad 2) ---")

# Create a pipeline with PolynomialFeatures and LinearRegression
poly_regressor_2 = Pipeline([
    ('poly', PolynomialFeatures(degree=2)),
    ('linear', LinearRegression())
])

poly_regressor_2.fit(X_train_reg_processed, y_train_reg)

# Make predictions on the test set
y_pred_poly_reg_2 = poly_regressor_2.predict(X_test_reg_processed)

# Evaluate the model
mae_poly_2 = mean_absolute_error(y_test_reg, y_pred_poly_reg_2)
mse_poly_2 = mean_squared_error(y_test_reg, y_pred_poly_reg_2)
rmse_poly_2 = np.sqrt(mse_poly_2)
r2_poly_2 = r2_score(y_test_reg, y_pred_poly_reg_2)

print("--- Rezultate Regresie Polinomială Salariu (grad 2) ---")
print(f"Mean Absolute Error (MAE): {mae_poly_2:.2f}")
print(f"Mean Squared Error (MSE): {mse_poly_2:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse_poly_2:.2f}")
print(f"R-squared (R2): {r2_poly_2:.4f}")

In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(y_test_reg, y_pred_poly_reg_2, alpha=0.3)

# Plot the ideal line (y=x)
plt.plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], 'r--', lw=2, label='Ideal Prediction Line (y=x)')

# Fit a simple linear model to the actual vs predicted points to get the prediction trend line
reg_line_model_poly_2 = LinearRegression()
reg_line_model_poly_2.fit(y_test_reg.values.reshape(-1, 1), y_pred_poly_reg_2)

# Generate points for the prediction trend line
x_vals_poly_2 = np.array([y_test_reg.min(), y_test_reg.max()]).reshape(-1, 1)
y_vals_reg_line_poly_2 = reg_line_model_poly_2.predict(x_vals_poly_2)

# Plot the prediction trend line (in orange)
plt.plot(x_vals_poly_2, y_vals_reg_line_poly_2, color='orange', lw=2, label='Model Prediction Trend Line')

plt.xlabel('Actual Salary')
plt.ylabel('Predicted Salary')
plt.title('Actual vs. Predicted Salary with Ideal and Model Trend Lines (Polynomial Regression, degree=2)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.show()

### Box Plots: Relația între Atributele Numerice și Clasa 'vacation'

In [ ]:
numerical_continuous_attributes = ['aggregated_score', 'salary', 'experience_years', 'skills_count', 'certifications', 'total_days_worked']

for attribute in numerical_continuous_attributes:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='vacation', y=attribute, data=education_train_set)
    plt.title(f'Distribution of {attribute} by Vacation Type')
    plt.xlabel('Vacation Type')
    plt.ylabel(attribute)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

### Box Plots: Relația între Atributele Categorice și 'salary'

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 'categoric_ordinal_attributes' is defined earlier in the notebook.
# Ensure 'salary' is in the dataset for plotting.

for attribute in categoric_ordinal_attributes:
    plt.figure(figsize=(12, 7))
    sns.boxplot(x=attribute, y='salary', data=education_train_set)
    plt.title(f'Distribution of Salary by {attribute}')
    plt.xlabel(attribute)
    plt.ylabel('Salary')
    plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for readability
    plt.tight_layout()
    plt.show()